# *Budget Estimation*

## *Data Attribution*


This dataset was constructed from 145 Israeli Ministry of Education research grant records. Each row represents one funded research proposal, extracted from a "golden triangle" of three source documents per proposal:

1. **Call for proposals** (קול קורא) — the funding call under which the proposal was submitted
2. **Research proposal** (הצעת מחקר) — the PI's submitted research design
3. **Budget breakdown** (פילוח תקציבי) — the itemized budget allocation for the proposal

Structured fields (25 attributes per record, including study design parameters, methodology flags, and budgeted person-work) were extracted from each triangle using Google NotebookLM, then consolidated, validated, and merged into a single structured table through manual QA.


## *Data Dictionary*



Field definitions as specified in the extraction prompt given to NotebookLM.

| Column | Type | Description |
|---|---|---|
| `call_title` | string | Title of the call for proposals (קול קורא) |
| `pi_name` | string | Name of the principal investigator(s) |
| `proposal_title` | string | Title of the submitted research proposal |
| `duration_months` | int | Research duration in months (`-1` = unknown) |
| `site_count` | int | Number of research sites / participating schools (`-1` = unknown) |
| `subject_count` | int | Number of research subjects / participants (`-1` = unknown) |
| `target_populations_count` | int | Number of distinct target populations (`-1` = unknown) |
| `data_waves_count` | int | Number of data collection waves (`-1` = unknown) |
| `has_surveys` | binary | Uses questionnaires/surveys (1 = yes, 0 = no, -1 = unknown) |
| `has_interviews` | binary | Uses interviews (1/0/-1) |
| `has_focus_groups` | binary | Uses focus groups (1/0/-1) |
| `has_observations` | binary | Uses observations (1/0/-1) |
| `has_tests` | binary | Uses tests/assessments (1/0/-1) |
| `collection_mode` | categorical | Primary data collection mode: `Online`, `Field`, `SurveyInstitute`, `Mixed`, `Unknown` |
| `tool_development_type` | categorical | Level of instrument development: `Existing`, `Adaptation`, `New`, `Validation`, `Unknown` |
| `languages_count` | int | Number of languages used (`-1` = unknown) |
| `requires_translation` | binary | Requires translation / cultural adaptation (1/0/-1) |
| `uses_admin_data` | binary | Uses administrative / research-room data (1/0/-1) |
| `requires_data_linking` | binary | Requires linking/merging of datasets (1/0/-1) |
| `is_complex_design` | binary | Complex design — control group / complex population (1/0/-1) |
| `budgeted_person_work` | int/float | Total budgeted amount for personnel work, in NIS (monetary value, not months; `-1` = unknown) |
| `has_external_services` | binary | Purchases external services (surveys, statistics, editing) (1/0/-1) |
| `deliverables_level` | categorical | Scope of deliverables: `Standard`, `Extended`, `Unknown` |
| `has_special_equipment` | binary | Includes dedicated equipment in budget (1/0/-1) |
| `is_multidisciplinary` | binary | Multidisciplinary research (1/0/-1) |

## *Setup*

### *Imports*

In [1]:
import pandas as pd
import numpy as np
import plotnine as pln
from scipy.stats import skew, kurtosis

### *Dataset Loading*

In [2]:
df = pd.read_csv(r"C:\Users\idowe\MyProjects\MOECSO\budget_estimation\research_table_with_sources.csv", encoding="utf-8-sig")

In [3]:
df.head()

,source_file,call_title,pi_name,proposal_title,duration_months,site_count,subject_count,target_populations_count,data_waves_count,has_surveys,...,languages_count,requires_translation,uses_admin_data,requires_data_linking,is_complex_design,budgeted_person_work,has_external_services,deliverables_level,has_special_equipment,is_multidisciplinary
0,אורטל_סלובדין_21.txt,קול קורא מס' 2021.11/27 למחקר בנושא החינוך בחב...,פרופ' אורטל סלובודין,"הארון הכפול: נוער להט""בי במערכת החינוך הערבית ...",18,80,80,-1,-1,0,...,3,-1,0,0,0,125280.0,1,Unknown,0,-1
1,אורטל_סלובדין_24.txt,קול קורא מס' 15/5.2024 למחקר בנושא חינוך בצל מ...,"פרופ' אורטל סלובודין, פרופ' הללי פינסון",חסמים באיתור ובהשתתפות תלמידים/ות מהחברה הבדוא...,18,8,100,2,1,0,...,2,1,1,0,1,134676.0,1,Extended,0,1
2,אורי_כהן_21.txt,קול קורא מס' 27/11.2021 למחקר בנושא החינוך בחב...,אורי כהן,יחסי גומלין בין מנהל החינוך ברשות המקומית לבתי...,18,25,65,3,-1,0,...,2,-1,0,0,0,248400.0,0,Unknown,0,-1
3,אורית_חזן_24.txt,"קול קורא מס' 27/8.24 למחקר בנושא ""קבלת החלטות ...",אורית חזן,קבלת החלטות בית ספריות מבוססות נתונים בקרב מנה...,16,25,175,1,2,1,...,1,0,0,0,1,217500.0,1,Standard,0,0
4,אורלי_ליפקה_24.txt,קול קורא מס 15/5.2024 למחקר בנושא חינוך בצל מש...,"אורלי ליפקה, תמר קציר",חוסן אישי ותחושת שלומות של מורים בחברה הערבית ...,18,-1,1255,2,2,1,...,1,1,0,0,1,135000.0,1,Extended,0,1


## *EDA - Explorary Data Analysis*

### *Structural analysis*

In [6]:
df.shape

(140, 26)

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   source_file               140 non-null    str    
 1   call_title                140 non-null    str    
 2   pi_name                   140 non-null    str    
 3   proposal_title            140 non-null    str    
 4   duration_months           140 non-null    int64  
 5   site_count                140 non-null    int64  
 6   subject_count             140 non-null    int64  
 7   target_populations_count  140 non-null    int64  
 8   data_waves_count          140 non-null    int64  
 9   has_surveys               140 non-null    int64  
 10  has_interviews            140 non-null    int64  
 11  has_focus_groups          140 non-null    int64  
 12  has_observations          140 non-null    int64  
 13  has_tests                 140 non-null    int64  
 14  collection_mode      